# AI Usage Protocol Notebook

Use this notebook to document relevant AI/LLM interactions for one milestone.

## Motivation and Relevance

This protocol is intended to support the documentation of AI usage in accordance with current university and NRW-related guidelines requiring transparency of AI-assisted work processes. In particular, students must disclose relevant uses of AI tools, including revisions, translations, text generation, image generation, analysis support, structuring support, or other AI-assisted scientific activities.

This protocol is not intended as a raw chat dump and not as a collection of successful AI interactions only.

Students must document all milestone-relevant AI interactions that substantially influenced scientific reasoning, understanding, decision-making, literature evaluation, argumentation, modeling, implementation, or interpretation — independent of whether the AI output was ultimately accepted, revised, or rejected.

Relevant interactions therefore include:

* accepted outputs,
* partially used outputs,
* rejected outputs,
* hallucinated or misleading outputs,
* exploratory interactions,
* and interactions that triggered validation or correction activities.

The protocol focuses on documenting scientific reasoning and validation processes, not only final integration into the submitted work.

For iterative or multi-step workflows, students should document the interaction as a sequence of connected reasoning episodes. This includes relevant follow-up prompts, refinements, manual modifications, validation loops, and corrections whenever they substantially influenced the scientific reasoning or outcome of the milestone work.
## Workflow

1. Execute the notebook initialization cell and fill in the general metadata fields for the milestone work that become visible afterwards.
2. For each scientifically relevant AI-supported reasoning episode, click **Add interaction**.
3. Document the relevant prompts, follow-up refinements, manual modifications, validation activities, evaluations, and reflections related to this reasoning episode.
4. Repeat this process for all relevant AI-supported reasoning episodes that influenced the milestone work — including accepted, revised, rejected, exploratory, or corrective usages.
5. Click **Save protocol JSON** to generate the machine-readable protocol file and the corresponding BibTeX entry (`.bib`).
6. Submit the completed notebook (`.ipynb`), the generated protocol (`.json`), and the generated BibTeX file (`.bib`) together with the milestone artifact.


In [3]:
import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError as e:
display(status_output)

    raise ImportError("ipywidgets is required. Install with: pip install ipywidgets") from e

PROTOCOL_VERSION = "0.2"

verification_levels = [
    "none",
    "plausibility-check",
    "secondary-source-check",
    "primary-source-check",
    "methodological-check",
    "empirical-check",
    "multi-source-validation"
]

evaluation_statuses = [
    "accepted",
    "partially-accepted",
    "rejected",
    "revised",
    "unresolved"
]

usefulness_levels = [
    "low",
    "medium",
    "high"
]

usage_types = [
    "idea-generation",
    "literature-search-support",
    "summarization",
    "terminology-clarification",
    "comparison-structuring",
    "argument-critique",
    "text-revision",
    "translation",
    "code-generation",
    "verification-support",
    "modeling-support",
    "requirements engineering-support",
    "other"
]

metadata = {
    "protocol_version": PROTOCOL_VERSION,
    "student_id": "",
    "course": "",
    "milestone": "",
    "topic": "",
    "entries": []
}

student_id = widgets.Text(description="Student ID:", placeholder="anonymous or matriculation-compatible ID")
course = widgets.Text(description="Course:", placeholder="Course/module name")
milestone = widgets.Text(description="Milestone:", placeholder="e.g. literature-analysis")
topic = widgets.Text(description="Topic:", placeholder="Your seminar/lab topic")

metadata_box = widgets.VBox([
    widgets.HTML("<h3>Protocol Metadata</h3>"),
    student_id,
    course,
    milestone,
    topic
])

entries_box = widgets.VBox([])
status_output = widgets.Output()

def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def make_interaction_widget(parent_id=None):
    entry_id = f"e-{uuid.uuid4().hex[:8]}"

    objective = widgets.Textarea(
        description="Objective:",
        placeholder="What scientific task did you try to solve?",
        layout=widgets.Layout(width="100%", height="70px")
    )

    ai_tool = widgets.Text(description="AI tool:", placeholder="e.g. ChatGPT, Claude, Perplexity")
    ai_model = widgets.Text(description="Model:", placeholder="e.g. GPT-5.5, unknown")

    prompt = widgets.Textarea(
        description="Prompt:",
        placeholder="Paste the relevant prompt here.",
        layout=widgets.Layout(width="100%", height="120px")
    )

    ai_output_summary = widgets.Textarea(
        description="AI output summary:",
        placeholder="Summarize the AI output. Do not paste long raw outputs unless required.",
        layout=widgets.Layout(width="100%", height="100px")
    )

    verification_level = widgets.RadioButtons(
        options=verification_levels,
        description="Verification level:",
        value="plausibility-check"
    )

    verification_actions = widgets.Textarea(
        description="Verification actions:",
        placeholder="What exactly did you check, and against which sources/evidence?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    evaluation_status = widgets.RadioButtons(
        options=evaluation_statuses,
        description="Status:",
        value="partially-accepted"
    )

    usefulness = widgets.RadioButtons(
        options=usefulness_levels,
        description="Usefulness:",
        value="medium"
    )

    issues = widgets.Textarea(
        description="Issues:",
        placeholder="List hallucinations, overgeneralizations, missing assumptions, wrong citations, etc.",
        layout=widgets.Layout(width="100%", height="90px")
    )

    used_in_work = widgets.Checkbox(description="Used in submitted work", value=False)
    section = widgets.Text(description="Section:", placeholder="e.g. Section 2.3")
    usage_type = widgets.Dropdown(options=usage_types, description="Usage type:", value="other")
    direct_text_reused = widgets.Checkbox(description="Direct text reused", value=False)

    reflection = widgets.Textarea(
        description="Reflection:",
        placeholder="What did you learn? How did this affect your understanding or scientific decision?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    remove_button = widgets.Button(description="Remove interaction", button_style="danger")

    box = widgets.VBox([
        widgets.HTML(f"<hr><h3>Interaction {entry_id}</h3>"),
        objective,
        widgets.HBox([ai_tool, ai_model]),
        prompt,
        ai_output_summary,
        widgets.HTML("<b>Verification</b>"),
        verification_level,
        verification_actions,
        widgets.HTML("<b>Evaluation</b>"),
        evaluation_status,
        usefulness,
        issues,
        widgets.HTML("<b>Integration</b>"),
        used_in_work,
        section,
        usage_type,
        direct_text_reused,
        widgets.HTML("<b>Reflection</b>"),
        reflection,
        remove_button
    ])

    box.entry_id = entry_id
    box.parent_id = parent_id
    box.fields = {
        "objective": objective,
        "ai_tool": ai_tool,
        "ai_model": ai_model,
        "prompt": prompt,
        "ai_output_summary": ai_output_summary,
        "verification_level": verification_level,
        "verification_actions": verification_actions,
        "evaluation_status": evaluation_status,
        "usefulness": usefulness,
        "issues": issues,
        "used_in_work": used_in_work,
        "section": section,
        "usage_type": usage_type,
        "direct_text_reused": direct_text_reused,
        "reflection": reflection
    }

    def remove_entry(_):
        children = list(entries_box.children)
        if box in children:
            children.remove(box)
            entries_box.children = tuple(children)

    remove_button.on_click(remove_entry)
    return box

def add_interaction(_):
    entries_box.children = tuple(list(entries_box.children) + [make_interaction_widget()])

def serialize_entry(box):
    f = box.fields

    verification_actions = [
        line.strip() for line in f["verification_actions"].value.splitlines()
        if line.strip()
    ]
    issues = [
        line.strip() for line in f["issues"].value.splitlines()
        if line.strip()
    ]

    return {
        "id": box.entry_id,
        "timestamp": now_iso(),
        "parent_id": box.parent_id,
        "objective": f["objective"].value.strip(),
        "ai_tool": {
            "name": f["ai_tool"].value.strip(),
            "model": f["ai_model"].value.strip()
        },
        "prompt": f["prompt"].value.strip(),
        "ai_output_summary": f["ai_output_summary"].value.strip(),
        "verification": {
            "actions": verification_actions,
            "level": f["verification_level"].value
        },
        "evaluation": {
            "status": f["evaluation_status"].value,
            "issues": issues,
            "usefulness": f["usefulness"].value
        },
        "integration": {
            "used_in_work": bool(f["used_in_work"].value),
            "section": f["section"].value.strip(),
            "usage_type": f["usage_type"].value,
            "direct_text_reused": bool(f["direct_text_reused"].value)
        },
        "reflection": f["reflection"].value.strip()
    }


def safe_bibtex_key(text):
    """Create a BibTeX-safe key component."""
    text = (text or "ai-usage-protocol").lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text or "ai-usage-protocol"

def escape_bibtex(value):
    """Escape a minimal set of characters for BibTeX fields."""
    value = str(value or "")
    return (
        value.replace("\\", "\\textbackslash{}")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("&", "\\&")
             .replace("%", "\\%")
             .replace("$", "\\$")
             .replace("#", "\\#")
             .replace("_", "\\_")
    )

def protocol_to_bibtex(protocol, json_filename=None):
    """
    Convert an AI usage protocol dictionary into a BibTeX entry.

    The generated entry cites the submitted AI usage protocol as a
    documented part of the student's scientific workflow. It does not
    replace the required disclosure of specific AI tools in the protocol.
    """
    exported_at = protocol.get("exported_at") or now_iso()
    year = exported_at[:4]
    student = protocol.get("student_id") or "anonymous"
    course_name = protocol.get("course") or "Course"
    milestone_name = protocol.get("milestone") or "Milestone"
    topic_name = protocol.get("topic") or "AI-assisted scientific work"
    version = protocol.get("protocol_version") or PROTOCOL_VERSION
    entries_count = len(protocol.get("entries", []))
    json_file = json_filename or "ai_usage_protocol.json"

    key_parts = [
        "ai-usage-protocol",
        safe_bibtex_key(student),
        safe_bibtex_key(milestone_name),
        year,
    ]
    bib_key = "-".join([part for part in key_parts if part])

    title = f"AI Usage Protocol for {milestone_name}: {topic_name}"
    note = (
        f"Machine-readable documentation of AI-assisted scientific work; "
        f"course: {course_name}; protocol version: {version}; "
        f"documented reasoning episodes: {entries_count}; "
        f"JSON file: {json_file}"
    )

    return f"""@misc{{{bib_key},
  author = {{{escape_bibtex(student)}}},
  title = {{{escape_bibtex(title)}}},
  year = {{{escape_bibtex(year)}}},
  howpublished = {{{escape_bibtex("Submitted AI usage protocol notebook and JSON file")}}},
  note = {{{escape_bibtex(note)}}}
}}"""


def save_protocol(_):
    metadata["student_id"] = student_id.value.strip()
    metadata["course"] = course.value.strip()
    metadata["milestone"] = milestone.value.strip()
    metadata["topic"] = topic.value.strip()
    metadata["exported_at"] = now_iso()
    metadata["entries"] = [serialize_entry(box) for box in entries_box.children]

    base_filename = f"ai_usage_protocol_{metadata['student_id'] or 'anonymous'}_{metadata['milestone'] or 'milestone'}"
    base_filename = base_filename.replace(" ", "_").replace("/", "-")
    json_filename = f"{base_filename}.json"
    bib_filename = f"{base_filename}.bib"

    Path(json_filename).write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    bibtex_entry = protocol_to_bibtex(metadata, json_filename=json_filename)
    Path(bib_filename).write_text(bibtex_entry + "\n", encoding="utf-8")

    with status_output:
        clear_output()
        print(f"Saved protocol JSON: {json_filename}")
        print(f"Saved BibTeX entry: {bib_filename}")
        print(f"Entries: {len(metadata['entries'])}")
        print()
        print("BibTeX entry:")
        print(bibtex_entry)

add_button = widgets.Button(description="Add interaction", button_style="primary")
save_button = widgets.Button(description="Save protocol JSON", button_style="success")

add_button.on_click(add_interaction)
save_button.on_click(save_protocol)

display(metadata_box)
display(widgets.HBox([add_button, save_button]))
display(entries_box)
display(status_output)


VBox()

Output()

## Recommended Use

For every milestone, start from a fresh copy of this notebook template.

Suggested rule for students:

- Do **not** document every trivial AI interaction.
- Do document every interaction that influenced a scientific decision, interpretation, comparison, argument, model, implementation, or submitted text.
- The most important part is not the prompt. The most important part is the **verification and evaluation**.

Suggested submission:

- completed notebook `.ipynb`
- generated protocol `.json`
- generated BibTeX file `.bib`
- milestone artifact, e.g. literature analysis, outline, model, code, or paper draft


## Controlled Vocabulary

### Verification level

- `none`: accepted without checking
- `plausibility-check`: checked only for plausibility
- `secondary-source-check`: checked against tutorials, summaries, lecture material, or secondary sources
- `primary-source-check`: checked against original papers, standards, documentation, or data
- `methodological-check`: checked the reasoning, method, assumptions, or formal correctness
- `empirical-check`: checked by running an experiment, model, tool, test, or implementation
- `multi-source-validation`: checked against multiple independent reliable sources

### Evaluation status

- `accepted`: used essentially as provided
- `partially-accepted`: used selectively
- `rejected`: discarded after evaluation
- `revised`: substantially corrected or rewritten
- `unresolved`: uncertainty remains


## Optional: Convert an Existing Protocol JSON to BibTeX

Use the following cell if you already have a generated protocol `.json` file and need to recreate the corresponding BibTeX entry.


In [ ]:
# Optional JSON-to-BibTeX converter
#
# Change this filename if needed, then run the cell.
# The generated .bib file can be included in the references of the scientific work.

json_protocol_file = "ai_usage_protocol_anonymous_milestone.json"

def convert_json_protocol_to_bibtex(json_file):
    json_path = Path(json_file)
    if not json_path.exists():
        raise FileNotFoundError(f"Protocol JSON file not found: {json_file}")

    protocol = json.loads(json_path.read_text(encoding="utf-8"))
    bibtex_entry = protocol_to_bibtex(protocol, json_filename=json_path.name)
    bib_path = json_path.with_suffix(".bib")
    bib_path.write_text(bibtex_entry + "\n", encoding="utf-8")
    print(f"Saved BibTeX entry: {bib_path}")
    print()
    print(bibtex_entry)

# Uncomment the next line after setting json_protocol_file correctly:
# convert_json_protocol_to_bibtex(json_protocol_file)
